# CRUX ML — Notebook 01: Data Fetch

**Zweck:** OHLCV-Daten + Funding Rates + Open Interest für alle Phase-1-Symbole von Binance & Dukascopy herunterladen, normalisieren und als Parquet cachen.

**Symbole Phase 1 (kostenlos):**
- Crypto: BTCUSDT, ETHUSDT, SOLUSDT (Binance)
- FX: EURUSD, GBPUSD, USDJPY (Dukascopy)
- Metalle: XAUUSD (Dukascopy)

**Output:** Parquet-Dateien in `data_cache/raw/<symbol>_<tf>.parquet`

**Laufzeit:** ~10 Minuten bei vollem 5-Jahres-Pull.

---

## Bedienung in Colab

1. `File → Open Notebook → GitHub → ecoNC/crux-ml → notebooks/01_fetch_data.ipynb`
2. `Runtime → Run all`
3. Beim ersten Lauf wird das Repo geklont und Dependencies installiert (~2 Min)
4. Daten landen im verbundenen Google Drive unter `MyDrive/crux-ml/data_cache/raw/`


## 1. Environment Setup

In [ ]:
# Detect environment (Colab vs local)
import sys, os
IS_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IS_COLAB}')
print(f'Python: {sys.version}')

In [ ]:
# Mount Google Drive (Colab only) for persistent data cache
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/crux-ml'
    os.makedirs(PROJECT_ROOT, exist_ok=True)
    os.chdir(PROJECT_ROOT)
    print(f'Project root: {PROJECT_ROOT}')
else:
    PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
    if not PROJECT_ROOT.endswith('crux-ml'):
        # Running from notebooks/ directly
        PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)
    print(f'Local project root: {PROJECT_ROOT}')

In [ ]:
# Clone repo into Drive if Colab and not present
if IS_COLAB and not os.path.exists(os.path.join(PROJECT_ROOT, 'core')):
    !git clone https://github.com/ecoNC/crux-ml.git /tmp/crux-ml
    !cp -r /tmp/crux-ml/. {PROJECT_ROOT}/
    print('Repo cloned to Drive')

In [ ]:
# Install dependencies
!pip install -q pandas pyarrow requests yfinance dukascopy-python tqdm 2>&1 | tail -1

In [ ]:
# Import project modules
sys.path.insert(0, PROJECT_ROOT)

from datetime import datetime, timezone
from pathlib import Path
from tqdm.notebook import tqdm
import pandas as pd

from core.config import (
    DATA_RAW, CRYPTO_SYMBOLS, FX_SYMBOLS, METAL_SYMBOLS,
    PRIMARY_TIMEFRAMES, HTF_CONTEXT_TIMEFRAMES,
    TRAIN_START, TEST_END,
)
from core.data import (
    fetch_klines, fetch_funding_rates, fetch_open_interest_hist,
    save_parquet, fetch_yahoo,
)

DATA_RAW.mkdir(parents=True, exist_ok=True)
print(f'Data cache: {DATA_RAW}')
print(f'Date range: {TRAIN_START.date()} → {TEST_END.date()}')

## 2. Crypto OHLCV (Binance) — BTC / ETH / SOL

In [ ]:
TIMEFRAMES_ALL = PRIMARY_TIMEFRAMES + HTF_CONTEXT_TIMEFRAMES

for symbol in tqdm(CRYPTO_SYMBOLS, desc='Crypto symbols'):
    for tf in TIMEFRAMES_ALL:
        out_path = DATA_RAW / f'{symbol}_{tf}.parquet'
        if out_path.exists():
            print(f'  skip {symbol} {tf} (cached)')
            continue
        try:
            df = fetch_klines(symbol, tf, TRAIN_START, TEST_END)
            if df.empty:
                print(f'  EMPTY {symbol} {tf}')
                continue
            save_parquet(df, out_path)
            print(f'  OK  {symbol} {tf}: {len(df):,} bars  {df.index[0].date()} → {df.index[-1].date()}')
        except Exception as e:
            print(f'  ERR {symbol} {tf}: {e}')

## 3. Crypto Funding Rates (Binance USD-M Futures)

In [ ]:
for symbol in tqdm(CRYPTO_SYMBOLS, desc='Funding rates'):
    out_path = DATA_RAW / f'{symbol}_funding.parquet'
    if out_path.exists():
        print(f'  skip {symbol} funding (cached)')
        continue
    try:
        df = fetch_funding_rates(symbol, TRAIN_START, TEST_END)
        if df.empty:
            print(f'  EMPTY {symbol} funding')
            continue
        save_parquet(df, out_path)
        print(f'  OK  {symbol} funding: {len(df):,} obs  {df.index[0].date()} → {df.index[-1].date()}')
    except Exception as e:
        print(f'  ERR {symbol} funding: {e}')

## 4. Crypto Open Interest (Binance Futures)

**Note:** Binance only provides ~30 days of OI history via the public endpoint.
For initial sanity check, we pull the last 30 days. For full historical OI we'd need
to either accumulate over time (monthly retraining helps) or use a third-party
archive in Phase 2.

In [ ]:
from datetime import timedelta
OI_LOOKBACK = datetime.now(timezone.utc) - timedelta(days=29)
OI_END = datetime.now(timezone.utc)

for symbol in tqdm(CRYPTO_SYMBOLS, desc='Open Interest'):
    out_path = DATA_RAW / f'{symbol}_oi.parquet'
    if out_path.exists():
        print(f'  skip {symbol} OI (cached)')
        continue
    try:
        df = fetch_open_interest_hist(symbol, '5m', OI_LOOKBACK, OI_END)
        if df.empty:
            print(f'  EMPTY {symbol} OI')
            continue
        save_parquet(df, out_path)
        print(f'  OK  {symbol} OI: {len(df):,} obs  {df.index[0]} → {df.index[-1]}')
    except Exception as e:
        print(f'  ERR {symbol} OI: {e}')

## 5. FX & Metals (Dukascopy)

**Note:** First run installs `dukascopy-python`. Large 5M pulls over 5 years can
take 3-5 minutes per symbol due to archive structure.

In [ ]:
try:
    from core.data.dukascopy_fetcher import fetch_dukascopy_ohlcv
    DUKAS_AVAILABLE = True
except ImportError as e:
    print(f'Dukascopy fetcher unavailable: {e}')
    DUKAS_AVAILABLE = False

In [ ]:
if DUKAS_AVAILABLE:
    for symbol in tqdm(FX_SYMBOLS + METAL_SYMBOLS, desc='FX/Metals'):
        for tf in TIMEFRAMES_ALL:
            out_path = DATA_RAW / f'{symbol}_{tf}.parquet'
            if out_path.exists():
                print(f'  skip {symbol} {tf} (cached)')
                continue
            try:
                df = fetch_dukascopy_ohlcv(symbol, tf, TRAIN_START, TEST_END)
                if df.empty:
                    print(f'  EMPTY {symbol} {tf}')
                    continue
                save_parquet(df, out_path)
                print(f'  OK  {symbol} {tf}: {len(df):,} bars  {df.index[0].date()} → {df.index[-1].date()}')
            except Exception as e:
                print(f'  ERR {symbol} {tf}: {e}')
else:
    print('Skipping FX/Metals — install dukascopy-python first.')

## 6. Macro Context (Yahoo: VIX, DXY, 10Y Yield)

In [ ]:
out_path = DATA_RAW / 'macro_daily.parquet'
if not out_path.exists():
    try:
        macro = fetch_yahoo(['VIX', 'DXY', 'TNX'], TRAIN_START, TEST_END, interval='1d')
        if not macro.empty:
            save_parquet(macro, out_path)
            print(f'OK macro daily: {len(macro):,} obs  {macro.index[0].date()} → {macro.index[-1].date()}')
            display(macro.tail())
        else:
            print('EMPTY macro')
    except Exception as e:
        print(f'ERR macro: {e}')
else:
    print('Macro daily cached')
    macro = pd.read_parquet(out_path)
    display(macro.tail())

## 7. Sanity Check — BTC 5M Sample

In [ ]:
btc_path = DATA_RAW / 'BTCUSDT_5m.parquet'
if btc_path.exists():
    btc = pd.read_parquet(btc_path)
    print(f'BTC 5M bars: {len(btc):,}')
    print(f'Date range: {btc.index[0]} → {btc.index[-1]}')
    print(f'Expected ~5 years ≈ 525,600 bars')
    print('\nSample tail:')
    display(btc.tail())
else:
    print('BTC 5M not yet fetched')

## 8. Summary

In [ ]:
parquet_files = sorted(DATA_RAW.glob('*.parquet'))
summary = []
for p in parquet_files:
    df = pd.read_parquet(p)
    summary.append({
        'file': p.name,
        'rows': len(df),
        'from': df.index[0] if len(df) else None,
        'to': df.index[-1] if len(df) else None,
        'size_MB': round(p.stat().st_size / 1024**2, 2),
    })
summary_df = pd.DataFrame(summary)
display(summary_df)
print(f'\nTotal cached files: {len(summary)}')
print(f'Total size: {summary_df["size_MB"].sum():.1f} MB')

---

## Nächster Schritt

→ `02_feature_engineering.ipynb`

Berechnet 30 ATR-normalisierte Features aus den gerade gecachten OHLCV-Daten.